# Grouped Planet Model — Adım Adım İşleyiş

Bu notebook `model_grouped_planet.py` içindeki **GroupedPlanetOrbitWarsPolicy** modelinin bir turn içinde nasıl karar verdiğini, her ara tensor'un shape ve anlamını göstererek anlatır.

Akış:

1. Observation nasıl üretilir, içinde ne var
2. `PlanetFeatureGroupProjector` — planet başına 6 semantik grup token
3. `group_embed` — her token'a grup kimliği eklemek
4. `GroupedPlanetRelationEncoder` — P·6 token üzerinde attention (full / induced)
5. PMA pooling ile grup özetleri ve global özet
6. Policy heads — `src_logits`, `tgt_logits (P×P)`, `ship_logits (P×P×bins)`, `value`
7. `act` ve `act_source` — joint source-target + ship bin seçimi
8. `evaluate_source_actions` — PPO'nun kullandığı log-prob/entropy
9. Rollout'taki ranked source-target seçimi, obs mutation, launch cap

Her kod hücresi CPU'da küçük bir model örneği ile çalışır; shape ve örnek değerler print edilir.

In [1]:
import os, sys, pathlib
repo_root = pathlib.Path().resolve().parent if pathlib.Path.cwd().name == 'docs' else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)
print('repo_root =', repo_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)

repo_root = /home/doruk/Documents/Kaggle/Orbit_Wars


torch 2.11.0+cu128


## 1. Observation üretimi

Kaggle `orbit_wars` env'inden canlı bir observation oluşturuyoruz. `build_observation` ham gözlemi model'in beklediği numpy tensorlarına çevirir.

Observation alanları:

- `planet_feats (P_MAX, PLANET_FEATURE_DIM)` — planet başına özellik vektörü
- `planet_mask (P_MAX,)` — aktif planetler
- `source_mask (P_MAX,)` — bizim olup launch yapabileceklerimiz
- `fleet_feats`, `fleet_mask` — havadaki fleet'ler (planet_only varyantında boş)
- `global_feats (GLOBAL_FEATURE_DIM,)` — tour sayısı, ships ratio, vs.
- `edge_bias (P_MAX+1, P_MAX+1, EDGE_FEATURE_DIM)` — planet-planet geometrik bias

In [2]:
from kaggle_environments import make
from obs_builder import build_observation, PLANET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM, P_MAX_DEFAULT

env = make('orbit_wars', debug=False)
env.configuration['randomSeed'] = 42
state = env.reset(num_agents=2)
state = env.step([[], []])  # bir step at, incoming fleet feature'ları için
raw_obs = state[0]['observation']
omega = raw_obs.get('angular_velocity', 0.0) or 0.0
obs = build_observation(raw_obs, player=0, angular_velocity=omega)

print('P_MAX            =', P_MAX_DEFAULT)
print('PLANET_FEATURE_DIM =', PLANET_FEATURE_DIM)
print('GLOBAL_FEATURE_DIM =', GLOBAL_FEATURE_DIM)
print('EDGE_FEATURE_DIM   =', EDGE_FEATURE_DIM)
print()
print('planet_feats  shape =', obs.planet_feats.shape)
print('planet_mask   shape =', obs.planet_mask.shape, ' active =', int(obs.planet_mask.sum()))
print('source_mask   shape =', obs.source_mask.shape, ' sources =', int(obs.source_mask.sum()))
print('global_feats  shape =', obs.global_feats.shape)
print('edge_bias     shape =', obs.edge_bias.shape)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy


... (10 outputs left out)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_python_repeated_pokerkit


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 1.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    snake


P_MAX            = 44
PLANET_FEATURE_DIM = 34
GLOBAL_FEATURE_DIM = 14
EDGE_FEATURE_DIM   = 6

planet_feats  shape = (44, 34)
planet_mask   shape = (44,)  active = 32
source_mask   shape = (44,)  sources = 1
global_feats  shape = (14,)
edge_bias     shape = (109, 109, 6)


## 2. `PlanetFeatureGroupProjector` — 6 semantik grup

Her planet'in ham feature vektörü 6 semantik gruba ayrılıp her grup için ayrı bir MLP ile `d_model` boyutuna yansıtılır:

| Grup | İçerdiği özellikler |
|---|---|
| owner | sahiplik (self/enemy/neutral/allied/empty) |
| economy | ships, production, radius, log_ships |
| geometry | r_sun, cos_theta, sin_theta |
| dynamics | is_static + rotation bilgisi |
| incoming | self/enemy incoming bucket ve ETA özetleri |
| value | capture value, net threat, defense deficit |

Çıktı shape: `(B, P, 6, d_model)`.

In [3]:
from model_grouped_planet import PlanetFeatureGroupProjector, GroupedPlanetOrbitWarsPolicy

D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2

proj = PlanetFeatureGroupProjector(planet_dim=PLANET_FEATURE_DIM, d_model=D_MODEL)
print('6 grup MLP sayısı:', proj.n_groups)

pf = torch.from_numpy(obs.planet_feats).unsqueeze(0).float()
grp = proj(pf)
print('planet_feats          ->', tuple(pf.shape))
print('group token tensor    ->', tuple(grp.shape), '  (B, P, G=6, d_model)')
print('mean / std            ->', f'{grp.mean().item():.3f} / {grp.std().item():.3f}')

6 grup MLP sayısı: 6
planet_feats          -> (1, 44, 34)
group token tensor    -> (1, 44, 6, 64)   (B, P, G=6, d_model)
mean / std            -> 0.000 / 0.999


## 3. `group_embed` — grup kimliği ekleme

Her grup token'ına "ben owner grubuyum / economy grubuyum / ..." demek için `nn.Embedding(6, d_model)` ağırlığı eklenir. Böylece P·6 token tek bir sete flatten edildiğinde attention her token'ın hangi gruba ait olduğunu bilir.

In [4]:
policy = GroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX_DEFAULT,
    planet_dim=PLANET_FEATURE_DIM,
    global_dim=GLOBAL_FEATURE_DIM,
    edge_dim=EDGE_FEATURE_DIM,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    attention_type='induced',
    n_inducing_points=8,
)
policy.eval()
print('group_embed.weight shape =', tuple(policy.encoder.group_embed.weight.shape))
print('her satır bir grubun öğrenilmiş kimliği.')

group_embed.weight shape = (6, 64)
her satır bir grubun öğrenilmiş kimliği.


## 4. Tokenizasyon: `_build_tokens`

6 grup tokeni `group_embed` ile toplanıp P·6 uzunluğunda tek bir sete flatten edilir. Bu set transformer/ISAB bloklarına girer.

Ayrıca:
- `_build_padding_mask` padded planet slotlarının tüm 6 grup token'ını mask'ler.
- `full` modda `_build_attn_bias` planet-planet edge bias'ını grup-grup çiftlerine kopyalar.
  `induced` modda edge bias kullanılmaz (N×N maliyetten kaçınmak için).

In [5]:
from model import pack_observations

batch = pack_observations([obs], device=torch.device('cpu'))
print('batch anahtarları:', list(batch.keys()))

enc = policy.encoder
flat_tok = enc._build_tokens(batch['planet_feats'], batch['fleet_feats'], batch['global_feats'])
print('P*6 token tensor   ->', tuple(flat_tok.shape))

pad_mask = enc._build_padding_mask(batch['planet_mask'], batch['fleet_mask'])
print('padding mask shape ->', tuple(pad_mask.shape), '  (aktif token sayısı =', int((~pad_mask).sum().item()), ')')

batch anahtarları: ['planet_feats', 'fleet_feats', 'global_feats', 'edge_bias', 'planet_mask', 'fleet_mask', 'source_mask']
P*6 token tensor   -> (1, 264, 64)
padding mask shape -> (1, 264)   (aktif token sayısı = 192 )


## 5. Encoder forward — attention ve pooling

`encode_planets` bütün boru hattını çalıştırır. Çıktı dict:

- `planet_tok (B, P, d_model)` — her planet için geç-fusion sonrası tek bir vektör
- `group_tokens (B, P, G, d_model)` — 6 grup token'ının contextual hali
- `group_summary (B, G, d_model)` — PMA pool'dan grup başına global özet
- `global_summary (B, d_model)` — grup özetlerinin fused hali + global_feats projection
- `planet_mask (B, P)` — padding için kopya

In [6]:
with torch.no_grad():
    enc_out = enc.encode_planets(
        planet_feats=batch['planet_feats'],
        fleet_feats=batch['fleet_feats'],
        global_feats=batch['global_feats'],
        edge_bias=batch['edge_bias'],
        planet_mask=batch['planet_mask'],
        fleet_mask=batch['fleet_mask'],
    )
for k, v in enc_out.items():
    if isinstance(v, torch.Tensor):
        print(f'{k:15s} -> shape={tuple(v.shape)}  mean={v.float().mean().item():+.3f}  std={v.float().std().item():.3f}')

group_tokens    -> shape=(1, 44, 6, 64)  mean=+0.000  std=0.853
planet_tokens   -> shape=(1, 44, 64)  mean=+0.000  std=0.853
group_summary_tokens -> shape=(1, 6, 64)  mean=-0.000  std=1.001
global_summary  -> shape=(1, 64)  mean=+0.000  std=1.008


## 6. Policy Heads

Encoder çıktılarından 4 ayrı head üretilir:

| Head | Çıktı | Shape | Anlam |
|---|---|---|---|
| `src_head` | launch score | (B, P) | her source'un launch eğilimi |
| `noop_head` | noop score | (B, P) + global | per-source no-op logiti |
| `tgt_head` | target pointer | (B, P, P) | source-target pair launch logiti |
| `ship_head` | ship bin | (B, P, P, n_bins) | (source, target) için gemi oranı |
| `value_head` | state value | (B,) | PPO critic |

Şimdi `policy.forward(...)` çıktılarını yazdıralım.

In [7]:
with torch.no_grad():
    out = policy(**batch)

print('src_logits         ->', tuple(out.src_logits.shape), '  (P+1: son slot global no-op compat)')
print('tgt_logits         ->', tuple(out.tgt_logits.shape), '  (B, P, P)')
print('ship_logits        ->', tuple(out.ship_logits.shape), '  (B, P, P, bins)')
print('value              ->', tuple(out.value.shape),      '  (B,)')
print('aux_owner_logits   ->', tuple(out.aux_owner_logits.shape), '  (B, P, n_owners)')

src_logits         -> (1, 45)   (P+1: son slot global no-op compat)
tgt_logits         -> (1, 44, 44)   (B, P, P)
ship_logits        -> (1, 44, 44, 5)   (B, P, P, bins)
value              -> (1,)   (B,)
aux_owner_logits   -> (1, 44, 5)   (B, P, n_owners)


### 6.1 `tgt_logits` — launch matrix

Bu, bizim source planet olan her slotun her hedefe gönderim skoru. Pad/self maskelenir.
Bir source slot için tüm hedef logit'lerini listeleyelim.

In [8]:
src_slots = [int(i) for i, m in enumerate(obs.source_mask) if m]
print('source slotlar:', src_slots)

if src_slots:
    s = src_slots[0]
    top_targets = torch.topk(out.tgt_logits[0, s], k=5)
    print(f'\nsource_slot={s} için en yüksek 5 target logiti:')
    for val, idx in zip(top_targets.values.tolist(), top_targets.indices.tolist()):
        print(f'  target_slot={idx:3d}  logit={val:+.3f}  planet_id={int(obs.planet_ids[idx])}')

source slotlar: [24]

source_slot=24 için en yüksek 5 target logiti:
  target_slot= 21  logit=-0.443  planet_id=21
  target_slot= 29  logit=-0.447  planet_id=29
  target_slot=  9  logit=-0.449  planet_id=9
  target_slot= 13  logit=-0.454  planet_id=13
  target_slot=  1  logit=-0.459  planet_id=1


### 6.2 `ship_logits` — target-conditioned bin seçimi

Aynı source için seçilen hedef değiştikçe ship bin tercihleri de değişebilir. Bu yüzden ship_head shape'i `(P, P, n_bins)`'dir.

In [9]:
from obs_builder import DEFAULT_SHIP_BINS
print('ship bin oranları:', DEFAULT_SHIP_BINS)

if src_slots:
    s = src_slots[0]
    for t in top_targets.indices.tolist()[:3]:
        bins = out.ship_logits[0, s, t]
        probs = torch.softmax(bins, dim=-1)
        print(f'src={s:3d} tgt={t:3d}  bin logits={bins.tolist()}  probs={[f"{p:.2f}" for p in probs.tolist()]}')

ship bin oranları: (0.1, 0.25, 0.5, 0.75, 1.0)
src= 24 tgt= 21  bin logits=[0.27350983023643494, 0.17810280621051788, 0.0708274394273758, -0.4694278836250305, 0.10638665407896042]  probs=['0.25', '0.22', '0.20', '0.12', '0.21']
src= 24 tgt= 29  bin logits=[0.27848947048187256, 0.17764490842819214, 0.07516549527645111, -0.4682435989379883, 0.1055244654417038]  probs=['0.25', '0.22', '0.20', '0.12', '0.21']
src= 24 tgt=  9  bin logits=[0.27392616868019104, 0.18581953644752502, 0.06931222975254059, -0.47497695684432983, 0.10844086110591888]  probs=['0.25', '0.23', '0.20', '0.12', '0.21']


## 7. `act` — joint source-target + ship bin

Grouped policy'de `act`, noop + tüm source-target çiftleri üzerinde tek bir Categorical kurup örnekler.
Seçilen pair'e göre ship bin ikinci bir Categorical'dan çekilir.

Deterministic modda en yüksek skorlu pair ve bin döner.

In [10]:
with torch.no_grad():
    action = policy.act(out, deterministic=True)
for k, v in action.items():
    if isinstance(v, torch.Tensor):
        print(f'{k:10s} -> {v.tolist()}')

src        -> [44]
tgt        -> [0]
ship_bin   -> [0]
is_noop    -> [True]
log_prob   -> [-2.6005477905273438]


## 8. `act_source` — fixed source için karar

Rollout aynı turn içinde aynı source'u birden fazla kez kullanabilmek için `act_source` kullanır. Burada source slot dıştan verilir; sadece noop + target dağılımı ve ship bin dağılımı kurulur.

In [11]:
if src_slots:
    s = src_slots[0]
    src_tensor = torch.tensor([s], dtype=torch.long)
    with torch.no_grad():
        a = policy.act_source(out, source_slot=src_tensor, deterministic=True)
    print(f'source={s} için deterministic action:')
    for k, v in a.items():
        if isinstance(v, torch.Tensor):
            print(f'  {k:10s} -> {v.tolist()}')

source=24 için deterministic action:
  src        -> [24]
  tgt        -> [0]
  ship_bin   -> [0]
  is_noop    -> [True]
  log_prob   -> [-2.6005477905273438]


## 9. `evaluate_source_actions` — PPO critic/actor logları

PPO, rollout'ta kaydedilen `(src, tgt, ship_bin, is_noop)` için log-prob ve entropy hesaplar. Yeni policy ile eski log-prob arasındaki oran clipped surrogate'e girer.

In [12]:
if src_slots:
    s = src_slots[0]
    src_t = torch.tensor([s], dtype=torch.long)
    tgt_t = torch.tensor([int(top_targets.indices[0])], dtype=torch.long)
    ship_t = torch.tensor([2], dtype=torch.long)
    noop_t = torch.tensor([False], dtype=torch.bool)
    ev = policy.evaluate_source_actions(out, src_t, tgt_t, ship_t, noop_t)
    for k, v in ev.items():
        if isinstance(v, torch.Tensor):
            print(f'{k:10s} -> {v.tolist()}')

log_prob   -> [-4.984952449798584]
entropy    -> [5.023861885070801]
value      -> [-0.1753542125225067]


## 10. Rollout'ta ranked source-target seçimi

`rl/rollout.py` grouped model için şu akışı uygular:

```text
available_pairs = (source_mask) & (planet_mask) & (src != tgt)
while launch_count < 8 and any(source):
    out = policy(obs_work)
    mask tgt_logits with available_pairs
    ranked = top 64 pairs by logit
    for pair in ranked:
        for ship_bin in ranked bins for (src,tgt):
            try decode_action -> valid launch?
                if yes: launch, update obs_work, disable (src,tgt) pair, break
```

Sabitler `rl/rollout.py` ve `rl/watch.py`'de `MAX_LAUNCHES_PER_PLAYER_TURN = 8`, `MAX_PAIR_CANDIDATES = 64`.

Launch sonrası observation şu şekilde mutate edilir:
- `planet_feats[src, PF_IDX_SHIPS] -= sent_ships / 100`
- `planet_feats[tgt, PF_IDX_INCOMING_SELF_B0 + eta_bucket] += sent_ships / 100`
- `source_mask[src] = (remaining_ships > 0.01)`
- `available_pairs[src, tgt] = False`

In [13]:
from rl.watch import _act_all_sources, MAX_LAUNCHES_PER_PLAYER_TURN, MAX_PAIR_CANDIDATES
print('cap:', MAX_LAUNCHES_PER_PLAYER_TURN, 'pair cap:', MAX_PAIR_CANDIDATES)

actions, infos = _act_all_sources(policy, raw_obs, 0, deterministic=True, device=torch.device('cpu'))
print('\ntürettiği env action sayısı:', len(actions))
for info in infos[:8]:
    print(f'  src={info["src_slot"]:3d}  tgt={info["tgt_slot"]:3d}  bin={info["ship_bin"]}  value={info["value"]:+.3f}')

cap: 8 pair cap: 64

türettiği env action sayısı: 1
  src= 24  tgt= 21  bin=4  value=-0.175


## 11. Özet

- Observation planet başına `PLANET_FEATURE_DIM` özellik, plus global + edge bias.
- `PlanetFeatureGroupProjector` her planeti 6 gruba ayırır → `(B, P, 6, d)`.
- `group_embed` her token'a grup kimliği katar; flatten ile `(B, P·6, d)` sete dönüşür.
- `GroupedPlanetRelationEncoder` transformer (full) veya ISAB (induced) blokları ile bu set üzerinde attention yapar. Full modda planet-planet edge bias tüm grup-grup çiftlerine kopyalanır.
- PMA-style learned seed pool → `group_summary (B, G, d)` ve `global_summary (B, d)`.
- Late-fusion ile `planet_tok (B, P, d)` tekrar tek bir planet vektörüne iner.
- Heads: `src_logits (B, P+1)`, `tgt_logits (B, P, P)`, `ship_logits (B, P, P, bins)`, `value (B,)`.
- Action semantics **pairwise**: karar noop + tüm (src, tgt) çiftleri üzerinde; ship bin seçilen pair'e koşullu.
- Rollout/watch: launch başına obs mutate, aynı source farklı target'a tekrar launch yapabilir, kullanılan pair aynı turn içinde kapanır, turn başına en fazla 8 launch + en fazla 64 pair denenir.
